In [ ]:
import random
import numpy as np
import pandas as pd
import re
import string
import tensorflow as tf
from tensorflow.keras.layers import LSTM, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.layers import TextVectorization

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
from zipfile import ZipFile

path_zip = "/content/contradictory-my-dear-watson.zip"

with ZipFile(path_zip, 'r') as zf:
  zf.extractall()

In [ ]:
root = "./"

train_df = pd.read_csv(root + "train.csv")
test_df = pd.read_csv(root + "test.csv")

In [ ]:
train_df.drop(["id", "lang_abv", "language"],axis=1,inplace=True)
test_df.drop(["id", "lang_abv", "language"],axis=1,inplace=True)

In [ ]:
max_len = max(len(text.split()) for text in train_df["premise"])
print(f'Max length of premise: {max_len}')

vectorizer = TextVectorization(output_mode='int', split='whitespace',
                               standardize='strip_punctuation',
                               max_tokens=10000,
                               output_sequence_length=max_len)

Max length of premise: 196


In [ ]:
def make_dataset(premise, hypothesis, label,
                 batch=32, buffer=10000):

    train_dataset = tf.data.Dataset.from_tensor_slices(((premise, hypothesis), label))
    dataset = train_dataset.batch(batch).shuffle(buffer).prefetch(tf.data.AUTOTUNE)

    val_size = int(len(dataset) * 0.2)

    # Split the dataset
    val_ds = dataset.take(val_size)
    train_ds = dataset.skip(val_size)

    return train_ds, val_ds

In [ ]:
def process_text(text):
    text = text.lower()

    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)

    text = re.sub(r'\d+', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
premise = train_df["premise"].apply(process_text)
hypothesis = train_df["hypothesis"].apply(process_text)
label = train_df["label"].to_numpy()

In [ ]:
vectorizer.adapt(np.concatenate([premise, hypothesis]))
vocab_size = vectorizer.vocabulary_size()
print(f'Vocabulary size: {vocab_size}')

Vocabulary size: 10000


In [ ]:
premise = vectorizer(premise)
hypothesis = vectorizer(hypothesis)

In [ ]:
train_ds, val_ds = make_dataset(premise, hypothesis, label)

In [ ]:
print(train_ds.element_spec)
print(val_ds.element_spec)

((TensorSpec(shape=(None, 196), dtype=tf.int64, name=None), TensorSpec(shape=(None, 196), dtype=tf.int64, name=None)), TensorSpec(shape=(None,), dtype=tf.int64, name=None))
((TensorSpec(shape=(None, 196), dtype=tf.int64, name=None), TensorSpec(shape=(None, 196), dtype=tf.int64, name=None)), TensorSpec(shape=(None,), dtype=tf.int64, name=None))


In [ ]:
class Base_Model(Model):
    def __init__(self, vocab_size, embedding_dim,
                 num_classes=3, name =""):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size,
                                                   embedding_dim)

        self.rnn1 = tf.keras.layers.Bidirectional(LSTM(64,
                                                       return_sequences=True,
                                                       return_state=False))
        self.batch_norm1 = BatchNormalization()

        self.rnn2 = tf.keras.layers.Bidirectional(LSTM(128,
                                                       return_sequences=True,
                                                       return_state=False))
        self.batch_norm2 = BatchNormalization()

        self.rnn3 = tf.keras.layers.Bidirectional(LSTM(256,
                                                       return_sequences=True,
                                                       return_state=False))
        self.batch_norm3 = BatchNormalization()

        self.dense = tf.keras.layers.Dense(num_classes,
                                           activation='softmax')

    def call(self, inputs, training=None):
        x = tf.cast(inputs, dtype=tf.int64)

        x = self.embedding(x)
        x = tf.reshape(x, [tf.shape(x)[0], -1, tf.shape(x)[-1]])

        x = self.rnn1(x)
        x = self.batch_norm1(x, training=training)

        x = self.rnn2(x)
        x = self.batch_norm2(x, training=training)

        x = self.rnn3(x)
        x = self.batch_norm3(x, training=training)

        x = self.dense(x)
        return x

In [ ]:
class Siameses(Model):
    def __init__(self, vocab_size, embedding_dim, num_classes=3, name="Siameses"):
        super(Siameses, self).__init__()

        self.shared_model = Base_Model(vocab_size, embedding_dim, name="Shared_Model")
        self.dense = tf.keras.layers.Dense(num_classes, activation='softmax')

    def call(self, inputs):
        sentence_a, sentence_b = inputs

        output_1 = self.shared_model(sentence_a)
        output_2 = self.shared_model(sentence_b)

        output_1 = tf.reduce_mean(output_1, axis=1)
        output_2 = tf.reduce_mean(output_2, axis=1)

        x = tf.concat([output_1, output_2], axis=-1)
        x = self.dense(x)

        return x

In [ ]:
model = Siameses(vocab_size=vocab_size,
                 embedding_dim=128)
input_a = tf.keras.Input(shape=(196,), dtype="int64")
input_b = tf.keras.Input(shape=(196,), dtype="int64")
model([input_a, input_b])
tf.keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss = tf.keras.losses.SparseCategoricalCrossentropy()
model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

In [ ]:
# Train the model
history = model.fit(train_ds, validation_data=val_ds, epochs=30)

Epoch 1/30
 10/304 ━━━━━━━━━━━━━━━━━━━━ 35:48 7s/step - accuracy: 0.3442 - loss: 1.1438